## Desafio 🎯

Neste desafio, você irá desenvolver um modelo de classificação de emoções na fala utilizando técnicas modernas de aprendizado de máquina, com foco em representações extraídas de modelos auto-supervisionados (SSL).

O modelo será treinado em múltiplos datasets de emoção na fala e avaliado em um dataset não visto durante o treinamento.

O objetivo é desenvolver modelos robustos que consigam generalizar para novos domínios, lidando com diferenças de idioma, qualidade de áudio e estilo de fala.

Se atentem principalmente à mudança de idioma: vocês conseguem identificar estratégias para melhorar a classificação em uma língua não vista durante o treinamento?

Além disso, desta vez vocês receberão datasets não padronizados. Será necessário tratar essas inconsistências, como diferenças de formato, taxa de amostragem e qualidade de áudio, de modo a construir um pipeline robusto e consistente para o treinamento dos modelos.

Explorem ao máximo estratégias baseadas em modelos auto-supervisionados (SSL)!!

Datasets treino e validação:

ANAD: https://drive.google.com/file/d/1o2Ea0g9lRZHFtlD7zYWtN0sueShTQZPs/view?usp=sharing


CaFE: https://drive.google.com/file/d/1o2Ea0g9lRZHFtlD7zYWtN0sueShTQZPs/view?usp=sharing


ASVP_ESD: https://drive.google.com/file/d/1-23Upua-L9nFCRASjdaBAw4cPW9927LW/view?usp=sharing


Dataset de teste:

https://drive.google.com/file/d/1gS3eKnflhsXN0j0kPaQ3Nzzd-rYL4P8X/view?usp=sharing

**1) Submeta os códigos desenvolvidos durante a atividade:**

O código desenvolvido está disponível no repositório GitHub: https://github.com/kaikicamilo/PAR-desafio2

Os arquivos principais são:
- `train.py`: pipeline completo de pré-processamento, treinamento e avaliação
- `calibrate.py`: calibração pós-treino via correção de prior para melhorar generalização cross-lingual

**2) Explique sucintamente como você resolveu o problema - considere os métodos utilizados para pré-processar os dados e para treinar o(s) modelo(s).**

**Pré-processamento dos dados:**

Os datasets apresentavam inconsistências significativas. O CaFE (francês) organiza amostras em subpastas por emoção e intensidade (Faible/Fort), com nomes de pastas em UTF-8 com problemas de encoding — tratado via mapeamento explícito de bytes. O ASVP_ESD segue convenção RAVDESS (emoção no 3º campo do nome do arquivo); identificou-se que o código '02' (calm) estava sendo mapeado incorretamente para neutral, inflando artificialmente essa classe com 1.066 amostras — esse código foi removido do treino. Todos os áudios foram reamostrados para 16 kHz via librosa e truncados/padded para 6 segundos. Foi utilizado `WeightedRandomSampler` para balancear as classes durante o treino (disgust tinha apenas 228 amostras vs. 2.456 de sad).

**Treinamento:**

O modelo `microsoft/wavlm-large` foi fine-tunado sobre os datasets CaFE e ASVP_ESD combinados (~7.250 amostras de treino após remoção do calm). As primeiras 16 das 24 camadas transformer foram congeladas para preservar representações gerais; as últimas 8 + cabeça de classificação foram treinadas com AdamW (lr=5e-6 para transformer, lr=5e-4 para cabeça), label smoothing=0.1 e mixed precision (AMP). Augmentação incluiu ruído gaussiano, ganho aleatório e pitch shift (±2 st).

**Calibração pós-treino:**

Foi aplicada correção de prior: os logits do modelo são ajustados por `logit[c] -= α × bias[c]`, onde o bias por classe é estimado no conjunto de validação como a diferença entre distribuição predita e verdadeira. O fator α ótimo (1.7) foi escolhido por grid search maximizando acurácia no val set.

**3) Explique sucintamente como se deu a escolha dos modelos utilizados. Qual foi o peso da análise do problema na escolha dos modelos?**

A escolha dos modelos foi fortemente guiada pela análise do problema: o desafio central é a **generalização cross-lingual** — treinar em inglês/francês e avaliar em bengalês (SUBESCO).

Foram testados em sequência:

1. **`wavlm-base-plus`** (pré-treinado em inglês): atingiu 77% de val acc mas colapsou para 31% no teste, prevendo quase exclusivamente 'neutral'. Descartado por falta de multilinguismo.

2. **`wav2vec2-large-xlsr-53`** (53 línguas): teoria boa, mas instabilidade na convergência e test acc de 31% — indicando que a representação multilingue sozinha não era suficiente.

3. **`wavlm-large`** (escolha final): projetado com objetivo de masked speech prediction com denoising, captura melhor features prosódicas (pitch, energia, ritmo) que são mais language-agnostic do que features fonéticas. Atingiu 60% de val acc já na época 1 (vs. 10 épocas para XLSR), confirmando superior capacidade de representação. A análise do problema — especialmente a importância de features prosódicas para transferência cross-lingual — foi determinante nessa escolha.

**3) Considerando o problema de classificação de emoção na fala para domínios não vistos, você considera que o problema foi resolvido com os métodos de classificação utilizados? Justifique sua resposta.**

O problema não foi completamente resolvido, mas foi significativamente endereçado.

A acurácia no dataset de teste (SUBESCO - bengalês) foi de **32% sem calibração** e **44% com calibração de prior**, contra uma baseline aleatória de 14,3% (7 classes uniformes). Isso representa aproximadamente 3× melhor que o aleatório, indicando que o modelo captura informação emocional cross-lingual real.

**Por que não foi completamente resolvido:**

Modelos SSL pré-treinados em inglês/francês aprendem features fonéticas específicas de cada língua. O bengalês possui fonologia, entonação e padrões prosódicos distintos, criando um shift de domínio que o fine-tuning em dados europeus não consegue cobrir completamente. Classes como disgust (recall 3%) e surprise (9%) foram praticamente não detectadas — sugerindo que essas emoções se manifestam acusticamente de forma muito diferente em bengalês.

**Estratégias que melhorariam o resultado:**

- Inclusão de dados de línguas altamente influenciadas pelo sânscrito no treino
- Domain adaptation adversarial (DANN) usando áudio bengalês não rotulado
- Self-training com pseudo-labels no conjunto de teste
- Features prosódicas explícitas (F0, energia) como entrada auxiliar, por serem mais language-agnostic

**4) Compartilhe o resultado obtido no dataset de teste. Quais conclusões podem ser retiradas a partir dos dados considerando o resultado?**

**Resultado final no dataset de teste (SUBESCO - bengalês, 7.000 amostras):**

| Métrica | Sem calibração | Com calibração de prior |
|---------|---------------|------------------------|
| Acurácia | 32,2% | **43,6%** |
| Macro F1 | 0,26 | **0,41** |

| Emoção | Precision | Recall | F1 |
|--------|-----------|--------|----|
| angry | 0,44 | 0,74 | 0,56 |
| disgust | 0,81 | 0,03 | 0,06 |
| fear | 0,57 | 0,33 | 0,42 |
| happy | 0,34 | 0,52 | 0,41 |
| neutral | 0,67 | 0,59 | 0,62 |
| sad | 0,34 | 0,45 | 0,39 |
| surprise | 0,39 | 0,39 | 0,39 |

**Conclusões:**

- Emoções com forte componente prosódico universal transferem bem: **angry** (recall 74%) e **neutral** (recall 59%) foram bem classificadas, pois raiva = alta energia/velocidade e neutralidade = padrão de referência são relativamente invariantes entre línguas.

- **Disgust** foi a emoção mais problemática (recall 3%): em bengalês, expressões de nojo podem ser mais sutis ou fonologicamente distintas, não correspondendo às features aprendidas.

- A **calibração de prior** foi crucial: sem ela, o modelo previa 'neutral' para 65% das amostras de teste (viés de domínio); com a correção, as predições ficaram muito mais balanceadas e a acurácia saltou de 32% para 44%.

- A gap entre val acc (~72%, inglês/francês) e test acc (44%, bengalês) evidencia o desafio fundamental de SER cross-lingual: métricas in-domain superestimam a capacidade de generalização.